# cALorie Notebook 38 — RepCount visual adjudication packet

This CPU notebook automatically fetches only the eight medium-confidence clips from the Notebook 37 post-run audit. It exports dense 4 FPS contact sheets and low-resolution silent review previews so the ambiguous visible-motion calls can be settled locally.

- No upload, file picker, widget, or manual label entry.
- No model scoring, training, calibration, or locked-benchmark access.
- The `stuN` token is treated as an unverified source code, never as a person identity.
- Full source videos are deleted; only 240p/8 FPS silent review derivatives are packaged.

Select a **CPU** runtime and use **Runtime → Run all**. One ZIP downloads at the end.


In [ ]:
%pip install -q --pre --upgrade --extra-index-url https://pypi.fury.io/lance-format pylance opencv-python-headless==4.11.0.86 pandas==2.2.3


In [ ]:
import hashlib, importlib.metadata, json, math, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

import cv2
import lance
import numpy as np
import pandas as pd
from google.colab import files
from IPython.display import display
from tqdm.auto import tqdm

SOURCE_COMMIT = "6b0dca6b814bf6b09f81206d6168a44ae6bd42aa"
RAW_ROOT = f"https://raw.githubusercontent.com/bariscelikk1/cALorie/{SOURCE_COMMIT}"
REVIEW_RELATIVE = "ml/data/repcount_development_v1/contact_sheet_visual_review_v1.csv"
REVIEW_SHA256 = "e3f4dd30c9b1f0817f90abd3f95e766394849dfee67d903c6e18b9257c0c231e"
LANCE_REVISION = "ae2e0b0496124d900d4e54a8e92f7ec15a292afd"
WORK = Path("/content/calorie_repcount_visual_adjudication_v1_work")
INPUTS, VIDEOS, SHEETS, PREVIEWS, REPORT = [WORK / x for x in ("inputs", "videos", "sheets", "previews", "report")]
for path in (INPUTS, VIDEOS, SHEETS, PREVIEWS, REPORT): path.mkdir(parents=True, exist_ok=True)

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""): digest.update(chunk)
    return digest.hexdigest()

review_path = INPUTS / "contact_sheet_visual_review_v1.csv"
urllib.request.urlretrieve(f"{RAW_ROOT}/{REVIEW_RELATIVE}", review_path)
assert sha256(review_path) == REVIEW_SHA256
review = pd.read_csv(review_path)
review["needs_second_review"] = review.needs_second_review.astype(str).str.lower().eq("true")
queue = review[review.needs_second_review].copy()
assert len(queue) == 8 and queue.video_id.is_unique
assert set(queue.video_id) == {"stu9_12", "stu1_53", "stu3_54", "stu5_52", "stu6_50", "stu8_49", "stu9_56", "stu2_67"}
print("PyLance:", importlib.metadata.version("pylance"))
display(queue[["video_id", "source_action", "visible_target", "visible_variant", "notes"]])


In [ ]:
scalar_columns = ["video_id", "source_name", "split", "action_type", "count", "cycle_bounds_json", "video_ext", "video_size_bytes"]
datasets, metadata_parts = {}, []
for split in ("train", "validation", "test"):
    uri = f"hf://datasets/lmms-lab-eval/repcounta-lance/data/{split}.lance"
    ds = lance.dataset(uri, storage_options={"hf_revision": LANCE_REVISION, "hf_enable_resolve_cache": "true"})
    frame = ds.to_table(columns=scalar_columns, with_row_id=True).to_pandas()
    frame["lance_split"] = split
    datasets[split] = ds
    metadata_parts.append(frame)
mirror = pd.concat(metadata_parts, ignore_index=True).rename(columns={"_rowid": "lance_row_id"})
selected = queue.merge(mirror, on="video_id", how="left", validate="one_to_one")
assert selected.lance_row_id.notna().all()
assert selected.source_action.eq(selected.action_type).all()

def parse_bounds(value):
    value = json.loads(value) if isinstance(value, str) else value
    numbers = []
    def walk(item):
        if isinstance(item, dict):
            for child in item.values(): walk(child)
        elif isinstance(item, (list, tuple, np.ndarray)):
            for child in item: walk(child)
        elif isinstance(item, (int, float)) and np.isfinite(item): numbers.append(float(item))
    walk(value)
    return sorted(x for x in numbers if x >= 0)

download_rows = []
for row in tqdm(selected.itertuples(index=False), total=len(selected), desc="Reading pinned blobs"):
    extension = str(row.video_ext or ".mp4")
    if not extension.startswith("."): extension = "." + extension
    target = VIDEOS / f"{row.video_id}{extension}"
    blob = datasets[row.lance_split].take_blobs("video_blob", ids=[int(row.lance_row_id)])[0]
    payload = blob.read()
    if hasattr(payload, "to_pybytes"): payload = payload.to_pybytes()
    target.write_bytes(payload)
    assert target.stat().st_size == int(row.video_size_bytes)
    cap = cv2.VideoCapture(str(target)); assert cap.isOpened(), target
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0); frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release(); assert fps > 0 and frames > 0
    bounds = parse_bounds(row.cycle_bounds_json); assert bounds
    start_s = max(0.0, min(bounds) / fps - 0.75)
    end_s = min(frames / fps, max(bounds) / fps + 0.75)
    assert end_s > start_s
    download_rows.append({"video_id": row.video_id, "video_path": str(target), "source_fps": fps, "source_frames": frames, "review_start_s": start_s, "review_end_s": end_s, "source_sha256": sha256(target), "lance_split": row.lance_split})
source = selected.merge(pd.DataFrame(download_rows), on=["video_id", "lance_split"], validate="one_to_one")
print("Fetched clips:", len(source), "source GB:", round(sum(Path(p).stat().st_size for p in source.video_path) / 1e9, 3))


In [ ]:
def render_dense_sheets(row, sample_fps=4.0, columns=4, rows=4):
    times = np.arange(row.review_start_s, row.review_end_s + 1e-9, 1 / sample_fps)
    cap = cv2.VideoCapture(row.video_path); pages = []
    for page_index in range(math.ceil(len(times) / (columns * rows))):
        page_times = times[page_index * columns * rows:(page_index + 1) * columns * rows]
        sheet = np.zeros((rows * 205, columns * 320, 3), dtype=np.uint8)
        for tile_index, time_s in enumerate(page_times):
            cap.set(cv2.CAP_PROP_POS_MSEC, float(time_s) * 1000)
            ok, frame = cap.read(); assert ok, (row.video_id, time_s)
            height, width = frame.shape[:2]; scale = min(320 / width, 180 / height)
            resized = cv2.resize(frame, (max(1, int(round(width * scale))), max(1, int(round(height * scale)))), interpolation=cv2.INTER_AREA)
            tile = np.zeros((205, 320, 3), dtype=np.uint8)
            x0 = (320 - resized.shape[1]) // 2; y0 = 25 + (180 - resized.shape[0]) // 2
            tile[y0:y0 + resized.shape[0], x0:x0 + resized.shape[1]] = resized
            cv2.putText(tile, f"{row.video_id} | {time_s:.2f}s", (7, 17), cv2.FONT_HERSHEY_SIMPLEX, 0.43, (0, 255, 255), 1, cv2.LINE_AA)
            x = (tile_index % columns) * 320; y = (tile_index // columns) * 205
            sheet[y:y + 205, x:x + 320] = tile
        output = SHEETS / f"{row.video_id}_{page_index:03d}.jpg"
        assert cv2.imwrite(str(output), sheet, [cv2.IMWRITE_JPEG_QUALITY, 88]); pages.append(output.name)
    cap.release(); return pages

def render_preview(row):
    output = PREVIEWS / f"{row.video_id}_review_240p.mp4"
    duration = row.review_end_s - row.review_start_s
    command = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-ss", f"{row.review_start_s:.6f}", "-i", row.video_path, "-t", f"{duration:.6f}", "-an", "-vf", "fps=8,scale=-2:240", "-c:v", "libx264", "-preset", "veryfast", "-crf", "30", "-pix_fmt", "yuv420p", str(output)]
    subprocess.run(command, check=True); assert output.stat().st_size > 0
    return output.name

packet_rows = []
for row in tqdm(source.itertuples(index=False), total=len(source), desc="Rendering review evidence"):
    pages = render_dense_sheets(row); preview = render_preview(row)
    packet_rows.append({"video_id": row.video_id, "source_action": row.source_action, "first_pass_target": row.visible_target, "first_pass_variant": row.visible_variant, "first_pass_confidence": row.confidence, "review_start_s": row.review_start_s, "review_end_s": row.review_end_s, "dense_sheet_pages": len(pages), "dense_sheet_pattern": f"sheets/{row.video_id}_*.jpg", "preview_path": f"previews/{preview}", "training_allowed": False, "adjudication_status": "pending_video_review"})
manifest = pd.DataFrame(packet_rows).sort_values("video_id")
manifest.to_csv(REPORT / "review_manifest.csv", index=False)
display(manifest)


In [ ]:
shutil.rmtree(VIDEOS)
assert not VIDEOS.exists()
shutil.copytree(SHEETS, REPORT / "sheets", dirs_exist_ok=True)
shutil.copytree(PREVIEWS, REPORT / "previews", dirs_exist_ok=True)
metadata = {"notebook": "38_colab_repcount_visual_adjudication_packet.ipynb", "source_commit": SOURCE_COMMIT, "repcount_lance_revision": LANCE_REVISION, "queued_clips": int(len(manifest)), "dense_sample_fps": 4.0, "preview_fps": 8, "preview_height": 240, "raw_source_video_packaged": False, "training_performed": False, "locked_benchmark_read": False, "person_identity_verified": False}
(REPORT / "run_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
(REPORT / "README.md").write_text("# RepCount visual adjudication packet v1\n\nEight medium-confidence rows from the Notebook 37 post-run audit. Review the silent 240p previews together with dense 4 FPS sheets. No source video, training, model score, or final adjudication is included. The stuN token is not a verified person identity.\n")
checks = []
for path in sorted(REPORT.rglob("*")):
    if path.is_file() and path.name != "SHA256SUMS.txt": checks.append(f"{sha256(path)}  {path.relative_to(REPORT)}")
(REPORT / "SHA256SUMS.txt").write_text("\n".join(checks) + "\n")
zip_path = WORK / "calorie_repcount_visual_adjudication_v1.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
    for path in sorted(REPORT.rglob("*")):
        if path.is_file(): archive.write(path, path.relative_to(REPORT))
with zipfile.ZipFile(zip_path) as archive: assert archive.testzip() is None
print("Ready:", zip_path.name, round(zip_path.stat().st_size / 1e6, 2), "MB", sha256(zip_path))
files.download(str(zip_path))
